# IS 362 Week 5: Airports and Wind

Data: `airports.csv` and `weather.csv` from [nycflights13 data-raw](https://github.com/hadley/nycflights13/tree/master/data-raw).


In [1]:
import pandas as pd

airports = pd.read_csv("airports.csv")
show = ["faa", "name", "lat", "lon", "tzone"]

## Question 1: Northernmost airport

I sorted by latitude in descending order.

In [2]:
airports.sort_values("lat", ascending=False)[show].head()

,faa,name,lat,lon,tzone
417,EEN,Dillant Hopkins Airport,72.270833,42.898333,NaN
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage


New Hampshires' latitude and longitude were swapped, so I fixed them.

In [3]:
airports.loc[airports["faa"] == "EEN", ["lat", "lon"]] = [42.898, -72.271]

airports.sort_values("lat", ascending=False)[show].head()

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage


**Answer: Wiley Post-Will Rogers Memorial Airport in Utqiagvik (Barrow), Alaska.**

Confirmed by the [Alaska Department of Transportation's master plan for the airport](https://aws.state.ak.us/OnlinePublicNotices/Notices/Attachment.aspx?id=154253), which says Utqiagvik, at 71 degrees north, "is the most northern city in the United States." BRW is the airport serving it, and it's the top airport in the data at about 71.3°N.

## Question 2: Easternmost airport

U.S. longitudes are negative, so the easternmost airport has the highest longitude.

In [4]:
airports.sort_values("lon", ascending=False)[show].head()

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
942,MYF,Montgomery Field,32.475900,117.759000,Asia/Chongqing
396,DVT,Deer Valley Municipal Airport,33.411700,112.457000,Asia/Chongqing
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York


The top four have positive longitudes, which shouldn't happen in the U.S.. Montgomery Field in San Diego and Deer Valley in Phoenix's time zones incorrectly say Asia. Eareckson Air Station in Alaska is past the 180° line, so its longitude is positive even though it's at the far west of the country, therefore I left it out. I fix the signs and only look at negative longitudes.

In [5]:
wrong_sign = airports["tzone"].str.startswith("Asia", na=False)
airports.loc[wrong_sign, "lon"] = -airports.loc[wrong_sign, "lon"]

airports[airports["lon"] < 0].sort_values("lon", ascending=False)[show].head()

,faa,name,lat,lon,tzone
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778,America/New_York


**Answer: Eastport Municipal Airport in Eastport, Maine.**

Confirmed by [General Aviation News](https://generalaviationnews.com/2014/08/20/eastport-municipal-airport-installs-awos/) which states "The airport is the most eastern airport in the United States".

## Question 3: Windiest NYC airport on Feb 12, 2013

In [6]:
weather = pd.read_csv("weather.csv")
feb12 = weather[(weather["month"] == 2) & (weather["day"] == 12)]

feb12.groupby("origin")["wind_speed"].agg(["mean", "max"])

,mean,max
origin,,
EWR,56.38822,1048.36058
JFK,14.38475,20.71404
LGA,14.96014,23.01560


EWR shows a max wind speed of 1,048 mph, even though the world record is about 253 mph. There is a singular outlier reading which I dismissed.

In [7]:
feb12[feb12["wind_speed"] > 100][["origin", "hour", "wind_speed"]]

,origin,hour,wind_speed
1009,EWR,3,1048.36058


In [8]:
feb12 = feb12[feb12["wind_speed"] < 100]

feb12.groupby("origin")["wind_speed"].agg(["mean", "max"]).sort_values("mean", ascending=False)

,mean,max
origin,,
LGA,14.960140,23.01560
JFK,14.384750,20.71404
EWR,13.258987,21.86482


**Answer: LaGuardia (LGA)**, with the highest average wind speed (about 15 mph) and the highest max (about 23 mph). EWR only looked windiest because of the outlier.


**Changes and assumptions**
- Fixed swapped latitude/longitude for Dillant Hopkins Airport (EEN).
- Added missing minus signs for Montgomery Field (MYF) and Deer Valley (DVT), the rows with Asian time zones.
- Easternmost only counts negative longitudes, so Shemya, AK is excluded.
- Removed the 1,048 mph EWR reading.